In [ ]:


%load_ext autoreload
%autoreload 2

In [439]:
import pandas as pd
import numpy as np

In [440]:
import matplotlib.pyplot as plt

In [441]:
feature_data = pd.read_csv("processed_data/pass_rusher_features.csv")

In [ ]:
feature_data.shape

In [ ]:
from baum_welch import *
from data_processing import *

In [ ]:
from ast import literal_eval

In [ ]:
positions = ["TE","T","C","G","FB","RB","WR"]
params = pd.read_csv("fitted_params.csv")
params["tau"] = params["tau"].apply(lambda x: np.array(literal_eval(x.replace("\n",","))) )

In [ ]:
poss_list = []
for key, data in feature_data.groupby(["gameId","playId"]):
    pos_dict = {}
    pos_dict["frames"] = data.frameId.values.tolist()
    pos_dict["gameId"] = key[0]
    pos_dict["playId"] = key[1]
    B, O, D = possession_to_voxel(data)
    k = O.shape[1]
    if k <= 1:
        continue
    player_pos_map = data.groupby(["nflId"]).apply(lambda x: x["officialPosition_pb"].unique()[0]).reset_index()
    merged_params = player_pos_map.merge(params, left_on = 0, right_on = "position")
    pos_dict["rusher_map"] = {index:val for index,val in enumerate(data.groupby("nflId_pr").groups.keys())}
    pos_dict["blocker_map"] = {index:val for index,val in enumerate(merged_params.nflId)}
    matchups = []
    for index, row in merged_params.iterrows():
        I, _ = expectation_matchup_step(row["tau"], row["sigma"], row["rho"], np.ones((k,1))/k, D[:,index,:], O, B )
        matchups.append(I)
    I_poss = np.stack(matchups, axis = 1)
    pos_dict["assignments"] = I_poss
    poss_list.append(pos_dict)

In [ ]:
all_pos_list = []
for poss in poss_list:
    gameId = poss["gameId"]
    playId = poss["playId"]
    poss_df_list = []
    rushers = poss["rusher_map"]
    blockers = poss["blocker_map"]
    for rusher in rushers:
        for blocker in blockers:
            assign_probs = poss["assignments"][:,blocker,rusher]
            prob_df = pd.DataFrame()
            prob_df["assignment_probs"] = assign_probs
            prob_df["nflId_pr"] = rushers[rusher]
            prob_df["nflId"] = blockers[blocker]
            poss_df_list.append(prob_df)
    poss_df = pd.concat(poss_df_list)
    poss_df["frameId"] = poss["frames"][0:len(poss_df)]
    poss_df["gameId"] = gameId
    poss_df["playId"] = playId
    all_pos_list.append(poss_df)
    

In [ ]:
assignment_data = pd.concat(all_pos_list)

In [442]:
# assignment_data.to_csv("assignment_data.csv",index=False)
assignment_data = pd.read_csv("assignment_data.csv")

In [443]:
assignment_data_agg = assignment_data.groupby(["nflId_pr","playId","frameId","gameId"]).apply(lambda x: {i:val for val, i in zip(x.assignment_probs,x.nflId)}).reset_index()

In [ ]:
assignment_data_agg.to_csv("assignment_data_agg.csv",index=False)

In [444]:
assignment_data_agg.rename(axis = 1, mapper = {0:"assignment_dict"}, inplace = True)

In [445]:
final_feature_data = assignment_data_agg.merge(feature_data.drop_duplicates(["time","nflId_pr"]))

In [450]:
rusher_encode_map = {index:val for index,val in enumerate(set(final_feature_data["nflId_pr"]))}

In [451]:
rusher_encode_inverse_map = {rusher_encode_map[index]:index for index in rusher_encode_map}

In [452]:
final_feature_data["rusher_id_model"] = final_feature_data["nflId_pr"].apply(lambda x: rusher_encode_inverse_map[x])

In [ ]:
from itertools import chain

In [446]:
blocker_encode_map = {index:val for index,val in enumerate(set(chain.from_iterable([list(item.keys()) for item in final_feature_data["assignment_dict"]])))}

In [449]:
blocker_encode_inverse_map = {blocker_encode_map[index]:index for index in blocker_encode_map}

In [453]:
final_feature_data_na = final_feature_data.dropna()
final_feature_data_na = final_feature_data[(~np.isinf(final_feature_data.strain_acceleration)) & (~np.isinf(final_feature_data.strain_rate)) ]

In [ ]:
final_feature_data_na

In [454]:
blocker_design_matrix = np.zeros((len(final_feature_data_na),len(blocker_encode_inverse_map)))

In [455]:
for i in range(len(final_feature_data_na)):
    assign_dict = final_feature_data_na.iloc[i]["assignment_dict"]
    blocker_design_matrix[i,[blocker_encode_inverse_map[val] for val in assign_dict]] = list(assign_dict.values()) 

In [457]:
pd.DataFrame(blocker_design_matrix).to_csv("blocker_rusher_design_matrix.csv")

In [458]:
final_feature_data_na.to_csv("strain_design_data.csv",index = False)

In [456]:
blocker_design_matrix.shape

(1150270, 534)

## Model Fitting

In [ ]:
from scipy.optimize import minimize, nnls

In [ ]:
from scipy.stats import expon

In [ ]:
ind_array = np.array([0,1,1])
val_array = np.array([5,19])
val_array[ind_array]

In [ ]:
def loss_function(coefs, acceleration, rusher_indices, X,n_rushers):
    rusher_coefs = coefs[0:n_rushers]
    blocker_coefs = coefs[n_rushers:]
    regression_loss = np.square(X.dot(blocker_coefs) + rusher_coefs[rusher_indices] - acceleration).sum()
    prior_loss = expon(1).logpdf(rusher_coefs).sum() + expon(1).logpdf(blocker_coefs).sum()
    return regression_loss - prior_loss

In [ ]:
rusher_design_matrix = pd.get_dummies(final_feature_data_na["rusher_id_model"]).values

In [ ]:
X_mat = -.5*final_feature_data_na["strain_rate"].values[:,np.newaxis]**2)*blocker_design_matrix

In [ ]:
minimize(loss_function, args = (final_feature_data_na["strain_rate"].values[:,np.newaxis],
                               final_feature_data_na["rusher_id_model"].values,
                               X_mat, 
                                len(rusher_encode_inverse_map)),
        x0 = np.ones((len(rusher_encode_inverse_map) + len(blocker_encode_inverse_map),)),
        bounds=[(0,np.infty) for _ in range(len(rusher_encode_inverse_map) + len(blocker_encode_inverse_map))])

### GMM for Blocked / Not blocked

In [1]:
import pandas as pd
import numpy as np
from itertools import chain
import ast

In [2]:
strain_data = pd.read_csv("strain_design_data.csv")

In [3]:
blocker_encode_map = {index:val for index,val in enumerate(set(chain.from_iterable([list(item.keys()) for item in [ast.literal_eval(item) for item in strain_data["assignment_dict"]]])))}

In [4]:
blocker_encode_inverse_map = {blocker_encode_map[index]:index for index in blocker_encode_map}

In [5]:
blocker_design_matrix = np.zeros((len(strain_data),len(blocker_encode_map)))

In [6]:
for i in range(len(strain_data)):
    assign_dict = ast.literal_eval(strain_data.iloc[i]["assignment_dict"])
    blocker_design_matrix[i,[blocker_encode_inverse_map[val] for val in assign_dict]] = list(assign_dict.values()) 

In [294]:
blocker_design_matrix_normalized = blocker_design_matrix / blocker_design_matrix.sum(axis=1,keepdims=True)

In [57]:
pd.DataFrame(blocker_design_matrix).to_csv("blocker_rusher_design_matrix.csv")

In [4]:
from sklearn.mixture import GaussianMixtureianMixture

In [5]:
gmm = GaussianMixture(n_components=3)

In [8]:
gmm.fit(strain_data["d2y_pr_qb"].values[:,np.newaxis])

GaussianMixture(n_components=3)

In [9]:
gmm.means_

array([[-1.80816931e+00],
       [ 1.79392383e+00],
       [-7.95854040e-04]])

In [10]:
gmm.precisions_

array([[[0.71231097]],

       [[0.7098447 ]],

       [[2.88181231]]])

In [13]:
block_state = gmm.predict_proba(strain_data["d2y_pr_qb"].values[:,np.newaxis])

In [16]:
strain_data["win_prob"] = block_state[:,1]

In [17]:
strain_data["lose_prob"] = block_state[:,0]

In [18]:
strain_data["block_prob"] = block_state[:,2]

In [20]:
strain_data.groupby(["nflId_pr"]).agg(Blocked_pct=("block_prob","mean"),Win_pct = ("win_prob","mean"),
                                     Lose_pct=("lose_prob","mean")).reset_index()

,nflId_pr,Blocked_pct,Win_pct,Lose_pct
0,33131,0.707026,0.151686,0.141288
1,34465,0.371386,0.285877,0.342737
2,34654,0.448592,0.467415,0.083993
3,34777,0.724277,0.141743,0.133979
4,35441,0.693662,0.158390,0.147948
...,...,...,...,...
693,53935,0.394038,0.227304,0.378658
694,53957,0.270492,0.418767,0.310741
695,53978,0.789566,0.113267,0.097167
696,53991,0.850353,0.054846,0.094800


In [21]:
players = pd.read_csv("data/players.csv")

nflId_pr  Blocked_pct   Win_pct  Lose_pct  nflId height  \
officialPosition                                                                
CB               508     48581     0.252305  0.569846  0.177849  48581    6-0   
                 256     44845     0.219894  0.665690  0.114416  44845    6-3   
                 310     45302     0.217911  0.690873  0.091217  45302   5-11   
                 228     43712     0.256005  0.732512  0.011484  43712    6-0   
                 351     46168     0.201311  0.789977  0.008712  46168    6-1   
DE               366     46205     0.472910  0.302038  0.225052  46205    6-2   
                 439     47832     0.453002  0.303946  0.243052  47832    6-3   
                 488     48145     0.517652  0.305037  0.177311  48145    6-3   
                 530     52456     0.395782  0.305996  0.298222  52456    6-3   
                 425     47799     0.415290  0.313214  0.271496  47799    6-5   
DT               177     43326     0.580058  0.200365  0.219577  43326    6-6   
                 5       35442     0.753043  0.200996  0.045961  35442    6-4   
                 82      41239     0.590533  0.215704  0.193762  41239    6-1   
                 58      40001     0.720214  0.226936  0.052851  40001    6-8   
                 599     52835     0.554966  0.231118  0.213916  52835    6-4   
FS               50      39941     0.401206  0.552312  0.046481  39941   5-10   
                 332     46123     0.305486  0.653597  0.040917  46123    6-1   
                 298     45020     0.283808  0.703296  0.012896  45020   5-10   
                 388     46349     0.178444  0.770451  0.051106  46349    6-1   
                 553     52512     0.159816  0.827729  0.012456  52512    6-0   
G                382     46267     0.889489  0.056425  0.054086  46267    6-3   
ILB              673     53615     0.614642  0.323586  0.061772  53615    6-4   
                 348     46157     0.460310  0.359293  0.180397  46157    6-3   
                 295     45008     0.557329  0.359600  0.083071  45008    6-1   
                 659     53575     0.282490  0.440574  0.276936  53575    6-1   
                 579     52623     0.208494  0.781621  0.009885  52623    6-1   
LB               686     53681     0.329447  0.337425  0.333128  53681    6-4   
MLB              30      37317     0.601019  0.220423  0.178557  37317    6-0   
                 249     44833     0.560147  0.241982  0.197871  44833    6-1   
                 497     48482     0.669840  0.262444  0.067716  48482    6-0   
                 486     48117     0.267578  0.369843  0.362579  48117    6-2   
                 587     52649     0.406389  0.445264  0.148347  52649    6-1   
NT               151     42559     0.734560  0.131906  0.133534  42559    6-2   
                 124     42382     0.758464  0.139789  0.101747  42382    6-3   
                 3       34777     0.724277  0.141743  0.133979  34777    6-3   
                 417     47786     0.736903  0.144612  0.118485  47786    6-3   
                 394     46487     0.690581  0.164202  0.145218  46487   5-11   
OLB              691     53910     0.419305  0.383288  0.197407  53910    6-2   
                 386     46313     0.395520  0.386942  0.217537  46313    6-4   
                 472     47947     0.430810  0.408359  0.160831  47947    6-4   
                 505     48554     0.408097  0.449678  0.142224  48554    6-1   
                 468     47938     0.182634  0.494893  0.322473  47938    6-1   
RB               672     53612     0.372742  0.273021  0.354237  53612    5-9   
SS               578     52607     0.429961  0.516102  0.053937  52607    6-2   
                 205     43407     0.206663  0.521015  0.272322  43407    6-0   
                 127     42389     0.308118  0.537155  0.154727  42389    6-1   
                 168     43303     0.187040  0.574880  0.238079  43303   5-10   
                 428     47804     0.314654  0.595824

In [135]:
snaps = strain_data.groupby(["nflId_pr","playId","gameId"]).agg(num_snaps = len)

In [138]:
snaps = strain_data.drop_duplicates(["nflId_pr","playId","gameId"])[["nflId_pr","playId","gameId"]].groupby("nflId_pr").agg(num_snaps=("playId",len)).reset_index()

In [43]:
strain_data.groupby(["nflId_pr"]).agg(Blocked_pct=("block_prob","mean"),Win_pct = ("win_prob","mean"),
                                     Lose_pct=("lose_prob","mean")).reset_index().merge(players, left_on = ["nflId_pr"], right_on = ["nflId"]).merge(snaps).groupby("officialPosition").apply(lambda x: x[(x.num_snaps >=90) & (x.officialPosition=="OLB")].sort_values("Win_pct",ascending=False).head(20))

nflId_pr  Blocked_pct   Win_pct  Lose_pct  nflId height  \
officialPosition                                                                
OLB              18      37075     0.355251  0.371449  0.273300  37075    6-3   
                 255     44842     0.380351  0.324599  0.295050  44842    6-4   
                 123     42381     0.399933  0.310108  0.289959  42381    6-5   
                 37      38551     0.422162  0.306344  0.271494  38551    6-5   
                 328     46110     0.406360  0.303396  0.290244  46110    6-2   
                 545     52492     0.418038  0.298706  0.283255  52492    6-5   
                 634     53479     0.464672  0.297088  0.238239  53479    6-2   
                 250     44834     0.430677  0.296374  0.272949  44834    6-3   
                 165     43298     0.391875  0.295250  0.312875  43298    6-5   
                 26      37145     0.434824  0.294809  0.270367  37145    6-3   
                 162     43292     0.434280  0.293941  0.271779  43292    6-5   
                 329     46117     0.432406  0.288779  0.278815  46117    6-2   
                 243     44825     0.399092  0.287994  0.312914  44825    6-1   
                 113     42346     0.478127  0.285371  0.236501  42346    6-3   
                 131     42401     0.445875  0.282414  0.271710  42401    6-3   
                 533     52468     0.429238  0.281666  0.289096  52468    6-2   
                 551     52510     0.420041  0.279342  0.300618  52510    6-4   
                 467     47934     0.468417  0.278801  0.252783  47934    6-4   
                 423     47795     0.416321  0.278184  0.305495  47795    6-5   
                 260     44859     0.431516  0.277621  0.290863  44859    6-3   

                      weight   birthDate               collegeName  \
officialPosition                                                     
OLB              18      250  1989-03-26             Texas A&amp;M   
                 255     252  1994-10-11                 Wisconsin   
                 123     265  1992-11-17         Mississippi State   
                 37      265  1990-02-27                  Syracuse   
                 328     250  1996-06-05            Boston College   
                 545     252  1998-08-25                   Alabama   
                 634     250         NaN                       NaN   
                 250     250  1995-03-06                  Missouri   
                 165     240  1992-09-08                   Georgia   
                 26      270  1989-01-21                   Georgia   
                 162     280  1995-07-11                Ohio State   
                 329     251  1996-12-28       Southern California   
                 243     235  1994-09-22                    Temple   
                 113     255  1994-08-03                   Florida   
                 131     260  1991-03-13                  Missouri   
                 533     250  1998-09-18                  Michigan   
                 551     242  1997-08-07  North Carolina-Charlotte   
                 467     242  1995-07-01                 Wisconsin   
                 423     277  1997-12-03                  Michigan   
                 260     240  1995-05-23                   Houston   

                     officialPosition        displayName  num_snaps  
officialPosition                                                     
OLB              18               OLB         Von Miller        158  
                 255              OLB          T.J. Watt        154  
                 123              OLB      Preston Smith        136  
                 37               OLB     Chandler Jones        163  
                 328              OLB      Harold Landry        220  
                 545              OLB      Terrell Lewis        154  
                 634              OLB      Azeez Ojulari        166  
                 250              OLB     Charles Harris        144 

In [62]:
stacked_transitions = strain_data.groupby(["nflId_pr","playId","gameId"]).apply(lambda x: x[["block_prob","win_prob","lose_prob"]].values).reset_index()

In [63]:
stacked_transitions.rename(axis = 1, mapper = {0:"time_assignments"}, inplace = True)

In [66]:
np.argmax(stacked_transitions.iloc[0]["time_assignments"],axis = 1)

array([0, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [85]:
def transition_calculator(x):
    state_array = np.argmax(x,axis = 1)
    transition_matrix = np.zeros((3,3))
    n = len(x)
    for i in range(n-1):
        cur_state = state_array[i]
        next_state = state_array[i+1]
        transition_matrix[cur_state,next_state] += 1
    return transition_matrix
    
    

In [86]:
stacked_transitions["transitions"] = stacked_transitions["time_assignments"].apply(lambda x: transition_calculator(x))

In [87]:
stacked_transitions

,nflId_pr,playId,gameId,time_assignments,transitions
0,33131,55,2021091300,"[[0.9071925984633156, 0.047145242423870505, 0....","[[16.0, 0.0, 1.0], [0.0, 0.0, 0.0], [1.0, 0.0,..."
1,33131,55,2021092602,"[[0.8586148908596849, 0.11160134027798214, 0.0...","[[24.0, 1.0, 1.0], [1.0, 6.0, 0.0], [1.0, 0.0,..."
2,33131,55,2021100311,"[[0.7765425062405933, 0.19598509087289456, 0.0...","[[6.0, 3.0, 0.0], [3.0, 8.0, 0.0], [0.0, 0.0, ..."
3,33131,56,2021102400,"[[0.8612319793097432, 0.031037140117009684, 0....","[[50.0, 0.0, 1.0], [0.0, 0.0, 0.0], [1.0, 0.0,..."
4,33131,63,2021101701,"[[0.789306984395422, 0.028846219158646963, 0.1...","[[25.0, 0.0, 1.0], [0.0, 0.0, 0.0], [1.0, 0.0,..."
...,...,...,...,...,...
36105,53991,311,2021101009,"[[0.875594821020664, 0.032215903762849475, 0.0...","[[21.0, 0.0, 1.0], [0.0, 0.0, 0.0], [1.0, 0.0,..."
36106,53991,810,2021101009,"[[0.9047576043208755, 0.040704980100611576, 0....","[[31.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0,..."
36107,53991,1180,2021101009,"[[0.9071952403291498, 0.04689426907670732, 0.0...","[[37.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0,..."
36108,53999,2769,2021100312,"[[0.8949556926941477, 0.0704876865800428, 0.03...","[[28.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0,..."


In [90]:
from collections import defaultdict
from operator import methodcaller
def merge_transitions(x):
    total_transitions = np.stack(x.values.tolist(),axis = 1).sum(axis=1)
    total_transitions /= total_transitions.sum(axis = 1, keepdims = True)
    return total_transitions

In [93]:
block_shed_ability = stacked_transitions.groupby("nflId_pr").apply(lambda x: merge_transitions(x.transitions)).reset_index()

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_91851/2675831523.py:5: RuntimeWarning: invalid value encountered in divide
  total_transitions /= total_transitions.sum(axis = 1, keepdims = True)


In [94]:
block_shed_ability.rename(axis = 1, mapper = {0:"transition_matrix"}, inplace = True)

In [97]:
block_shed_ability["shed_probability"] = block_shed_ability["transition_matrix"].apply(lambda x: x[0,1])

In [98]:
block_shed_ability["winning_power"] = block_shed_ability["transition_matrix"].apply(lambda x: x[1,1])

In [106]:
block_shed_ability["block-ability"] = block_shed_ability["transition_matrix"].apply(lambda x: x[0,0])

In [117]:
block_shed_ability.merge(players, left_on = ["nflId_pr"], right_on = ["nflId"]).merge(snaps).groupby("officialPosition").apply(lambda x: x[(x.num_snaps >=90) & (x.officialPosition=="DE")][["winning_power","displayName","num_snaps","shed_probability","block-ability"]].sort_values("shed_probability",ascending=False).head(20))

winning_power         displayName  num_snaps  \
officialPosition                                                     
DE               244       0.790323       Derek Barnett        159   
                 530       0.859454      Darrell Taylor        112   
                 425       0.851996         Brian Burns        201   
                 277       0.839978    Trey Hendrickson        213   
                 240       0.835199       Myles Garrett        206   
                 364       0.805461          Josh Sweat        166   
                 625       0.824798    Gregory Rousseau        102   
                 531       0.832898        A.J. Epenesa         91   
                 296       0.834548  Al-Quadin Muhammad        177   
                 39        0.846154    Whitney Mercilus        129   
                 623       0.825921          Kwity Paye        111   
                 33        0.841866       Mario Addison        113   
                 65        0.837989         Alex Okafor        123   
                 133       0.809589         Frank Clark        114   
                 8         0.822875   Jason Pierre-Paul        182   
                 11        0.831818        Jerry Hughes        133   
                 416       0.818871           Nick Bosa        168   
                 613       0.835946          Bryce Huff        135   
                 136       0.837707     Danielle Hunter        198   
                 190       0.846748     Yannick Ngakoue        195   

                      shed_probability  block-ability  
officialPosition                                       
DE               244          0.102257       0.789498  
                 530          0.091603       0.820264  
                 425          0.090474       0.824200  
                 277          0.078821       0.842659  
                 240          0.078790       0.845887  
                 364          0.078119       0.844141  
                 625          0.077626       0.843037  
                 531          0.076250       0.841875  
                 296          0.075949       0.844937  
                 39           0.073399       0.875000  
                 623          0.072778       0.847778  
                 33           0.072629       0.847154  
                 65           0.071322       0.868747  
                 133          0.070994       0.853448  
                 8            0.070299       0.864447  
                 11           0.070208       0.846651  
                 416          0.070040       0.858453  
                 613          0.069980       0.849899  
                 136          0.069705       0.850462  
                 190          0.069010       0.858073

In [222]:
n_rushers = max(strain_data["rusher_id_model"])+1

In [223]:
rusher_encoder_map = {index:val for index,val in zip(strain_data["rusher_id_model"],strain_data["nflId_pr"])}

In [311]:
rusher_coefs = pd.DataFrame(x_k[0:n_rushers][:,0], columns=["estimate"])

In [312]:
rusher_coefs["nflId_pr"] = rusher_coefs.index.map(lambda x: rusher_encoder_map[x])

In [112]:
players = pd.read_csv("data/players.csv")

In [315]:
rusher_coefs.merge(players,right_on = "nflId",left_on = "nflId_pr").merge(snaps).groupby("officialPosition").apply(lambda x: x[(x.officialPosition == "DT") & (x.num_snaps >= 90)].sort_values("estimate",ascending = False).head(20))

estimate  nflId_pr  nflId height  weight   birthDate  \
officialPosition                                                             
DT               336  0.073162     46138  46138    6-3     311  1995-04-20   
                 582  0.070041     38542  38542    6-4     310  1990-12-13   
                 216  0.068020     47802  47802    6-4     300  1997-07-28   
                 613  0.053621     38667  38667    6-5     290  1990-01-11   
                 94   0.052583     43338  43338    6-3     306  1992-12-16   
                 689  0.045572     45011  45011    6-0     321  1995-01-19   
                 22   0.040745     53467  53467    6-5     310         NaN   
                 36   0.038393     37104  37104    6-5     295  1989-05-06   
                 308  0.037031     46082  46082    6-3     320  1997-05-27   
                 520  0.034842     48544  48544    6-3     300  1995-10-20   
                 84   0.032914     43326  43326    6-6     308  1994-07-03   
                 160  0.028699     43436  43436    6-4     291  1993-03-31   
                 136  0.017480     41341  41341    6-4     322  1991-12-27   
                 650  0.014873     53073  53073    6-3     290  1997-02-28   
                 163  0.009842     43441  43441    6-3     310  1994-01-11   
                 238  0.009731     35562  35562    6-4     330  1987-03-25   
                 107  0.006893     43356  43356    6-2     308  1995-04-08   
                 621  0.006317     44829  44829    6-3     300  1995-01-16   
                 344  0.003512     46144  46144    6-1     310  1996-05-09   
                 222  0.000000     47811  47811    6-6     295  1996-10-08   

                                collegeName officialPosition  \
officialPosition                                               
DT               336   North Carolina State               DT   
                 582      Mississippi State               DT   
                 216      Mississippi State               DT   
                 613              Tennessee               DT   
                 94                 Alabama               DT   
                 689            Mississippi               DT   
                 22                     NaN               DT   
                 36              Ohio State               DT   
                 308                Alabama               DT   
                 520              Tennessee               DT   
                 84       Mississippi State               DT   
                 160               Maryland               DT   
                 136             Penn State               DT   
                 650  Florida International               DT   
                 163                 Temple               DT   
                 238        Louisiana State               DT   
                 107               Nebraska               DT   
                 621                Alabama               DT   
                 344          Florida State               DT   
                 222             Notre Dame               DT   

                            displayName  num_snaps  
officialPosition                                    
DT               336          B.J. Hill        132  
                 582       Fletcher Cox        175  
                 216    Jeffery Simmons        263  
                 613      Malik Jackson        193  
                 94         Jarran Reed        168  
                 689         D.J. Jones        100  
                 22   Christian Barmore        183  
                 36     Cameron Heyward        198  
                 308        Daron Payne        231  
                 520         Shy Tuttle         95  
                 84         Chris Jones        149  
                 160  Quinton Jefferson        160  
                 136       DaQuan Jones        133  
                 650         Teair Tart         90  
                 163     Matt Ioannidis        1

In [137]:
snaps.reset_index()[["nflId_pr"]]

frameId  assignment_dict  action  ball_snap  time  \
nflId_pr playId gameId                                                          
33131    55     2021091300       21               21      21         21    21   
                2021092602       44               44      44         44    44   
                2021100311       21               21      21         21    21   
         56     2021102400       53               53      53         53    53   
         63     2021101701       29               29      29         29    29   
...                             ...              ...     ...        ...   ...   
53991    311    2021101009       28               28      28         28    28   
         810    2021101009       32               32      32         32    32   
         1180   2021101009       38               38      38         38    38   
53999    2769   2021100312       31               31      31         31    31   
         2791   2021100312       24               24      24         24    24   

                            x_pr  y_pr  officialPosition  snap_time  nflId_qb  \
nflId_pr playId gameId                                                          
33131    55     2021091300    21    21                21         21        21   
                2021092602    44    44                44         44        44   
                2021100311    21    21                21         21        21   
         56     2021102400    53    53                53         53        53   
         63     2021101701    29    29                29         29        29   
...                          ...   ...               ...        ...       ...   
53991    311    2021101009    28    28                28         28        28   
         810    2021101009    32    32                32         32        32   
         1180   2021101009    38    38                38         38        38   
53999    2769   2021100312    31    31                31         31        31   
         2791   2021100312    24    24                24         24        24   

                            ...  d2y_pr  d2x_pr  pr_qb_norm  \
nflId_pr playId gameId      ...                               
33131    55     2021091300  ...      21      21          21   
                2021092602  ...      44      44          44   
                2021100311  ...      21      21          21   
         56     2021102400  ...      53      53          53   
         63     2021101701  ...      29      29          29   
...                         ...     ...     ...         ...   
53991    311    2021101009  ...      28      28          28   
         810    2021101009  ...      32      32          32   
         1180   2021101009  ...      38      38          38   
53999    2769   2021100312  ...      31      31          31   
         2791   2021100312  ...      24      24          24   

                            scalar_projection  d2y_pr_qb  d2x_pr_qb  d_ij  \
nflId_pr playId gameId                                                      
33131    55     2021091300                 21         21         21    21   
                2021092602                 44         44         44    44   
                2021100311                 21         21         21    21   
         56     2021102400                 53         53         53    53   
         63     2021101701                 29         29         29    29   
...                                       ...        ...        ...   ...   
53991    311    2021101009                 28         28         28    28   
         810    2021101009                 32         32         32    32   
         1180   2021101009                 38         38         38    38   
53999    2769   2021100312                 31         31         31    31   
         2791   2021100312                 24         24         24    24   

                            strain_rate  strain_acceleration  rusher_id_model  
nflId_pr playId ga

In [284]:
blocker_df = pd.DataFrame(x_k[len(rusher_encoder_map):],columns = ["estimate"])

In [285]:
blocker_df["nflId"] = blocker_df.index.map(lambda x: blocker_encode_map[x])

In [231]:
blocker_df.merge(players).groupby("officialPosition").apply(lambda x: x[x.officialPosition=="T"].sort_values("estimate",ascending=False).head(20))

estimate  nflId height  weight   birthDate  \
officialPosition                                                   
RB               248  0.356799  46096   5-11     220  1996-02-02   
                 107  0.289492  53623   5-11     221         NaN   
                 1    0.270709  45062   5-11     222  1994-09-16   
                 254  0.243771  46107   5-11     208  1997-08-03   
                 449  0.238261  52859   5-11     210  1996-02-28   
                 509  0.229706  44927   5-11     240  1995-09-16   
                 471  0.227196  52972    5-8     211  1997-10-06   
                 507  0.226759  44917    6-1     233  1995-05-05   
                 263  0.194859  39975    6-2     238  1991-03-17   
                 378  0.184135  48472    6-0     205  1995-08-17   
                 224  0.177992  47987   5-10     202  1998-08-07   
                 387  0.155494  42358    6-1     215  1993-04-13   
                 492  0.149311  44860    6-1     220  1996-07-24   
                 323  0.140198  52440    5-8     209  1999-04-11   
                 501  0.137391  42837   5-11     222  1993-05-15   
                 208  0.136981  47911    6-0     209  1997-04-30   
                 156  0.135743  45719   5-10     205  1994-05-04   
                 103  0.134298  41325    5-9     205  1992-05-03   
                 189  0.133609  47853    5-8     200  1997-08-19   
                 196  0.127525  47870   5-11     215  1997-02-11   

                              collegeName officialPosition  \
officialPosition                                             
RB               248      San Diego State               RB   
                 107                  NaN               RB   
                 1         Oklahoma State               RB   
                 254  Southern California               RB   
                 449                  NaN               RB   
                 509             Oklahoma               RB   
                 471          Mississippi               RB   
                 507           Pittsburgh               RB   
                 263            Tennessee               RB   
                 378           California               RB   
                 224                Miami               RB   
                 387            Wisconsin               RB   
                 492             Oklahoma               RB   
                 323      Louisiana State               RB   
                 501                Texas               RB   
                 208              Memphis               RB   
                 156            Wisconsin               RB   
                 103     Georgia Southern               RB   
                 189              Memphis               RB   
                 196              Alabama               RB   

                                displayName  
officialPosition                             
RB               248          Rashaad Penny  
                 107        Elijah Mitchell  
                 1             Chris Carson  
                 254           Ronald Jones  
                 449           Rodney Smith  
                 509          Samaje Perine  
                 471       Scottie Phillips  
                 507           James Conner  
                 263  Cordarrelle Patterson  
                 378          Patrick Laird  
                 224           Travis Homer  
                 387          Melvin Gordon  
                 492              Joe Mixon  
                 323  Clyde Edwards-Helaire  
                 501          Malcolm Brown  
                 208           Tony Pollard  
                 156        Dare Ogunbowale  
                 103        Jerick McKinnon  
                 189      Darrell Henderson  
                 196          Damien Harris

In [162]:
plays = pd.read_csv("data/pffScoutingData.csv")

In [167]:
snaps_block = plays[plays.pff_role == "Pass Block"].groupby("nflId").apply(lambda x: len(x.drop_duplicates(["gameId","playId"]))).reset_index()

In [168]:
snaps_block.rename(axis = 1, mapper = {0:"num_snaps"}, inplace=True)

In [292]:
snaps_block.merge(blocker_df).merge(players).groupby("officialPosition").apply(lambda x: x[(x.officialPosition=="T") & (x.num_snaps >= 90)].sort_values("estimate",ascending=False).head(20))

nflId  num_snaps  estimate height  weight   birthDate  \
officialPosition                                                              
T                451  52666        155  2.082355    6-5     318  1996-08-16   
                 429  52524        128  2.027822    6-6     305  1998-07-22   
                 447  52603        175  1.775740    6-5     290  1995-11-27   
                 175  43640        164  1.714519    6-6     313  1992-01-12   
                 165  43447        174  1.677853    6-5     324  1992-02-06   
                 31   39146        109  1.325996    6-5     310  1989-04-11   
                 14   37090        279  1.263189    6-8     325  1988-04-12   
                 474  53436        326  1.237296    6-6     325         NaN   
                 508  53557        230  1.133497    6-5     315         NaN   
                 462  52938        208  1.055611    6-6     310  1997-06-04   
                 271  46131        254  1.002017    6-7     297  1995-09-15   
                 478  53452         90  0.998265    6-5     314         NaN   
                 61   41237        182  0.971404    6-7     309  1991-07-22   
                 395  52418        149  0.882467    6-5     320  1999-05-17   
                 378  48181         93  0.862242    6-7     314  1996-01-02   
                 2    30869        217  0.853033    6-7     330  1981-12-12   
                 101  42377        306  0.824574    6-6     338  1993-06-23   
                 325  47794        258  0.717690    6-4     305  1997-11-17   
                 272  46134        104  0.679857    6-8     320  1995-10-21   
                 385  48455        328  0.649740    6-7     299  1995-12-19   

                                 collegeName officialPosition  \
officialPosition                                                
T                451                  Oregon                T   
                 429       St. John's, Minn.                T   
                 447             Wake Forest                T   
                 175    Southern Mississippi                T   
                 165          South Carolina                T   
                 31             Oregon State                T   
                 14                 Colorado                T   
                 474                     NaN                T   
                 508                     NaN                T   
                 462              Texas Tech                T   
                 271              Pittsburgh                T   
                 478                     NaN                T   
                 61                 Michigan                T   
                 395                 Alabama                T   
                 378           Virginia Tech                T   
                 2           Louisiana State                T   
                 101              Penn State                T   
                 325                 Alabama                T   
                 272  North Carolina A&amp;T                T   
                 385                    Iowa                T   

                              displayName  
officialPosition                           
T                451  Calvin Throckmorton  
                 429           Ben Bartch  
                 447        Justin Herron  
                 175          Rashod Hill  
                 165        Brandon Shell  
                 31          Mike Remmers  
                 14           Nate Solder  
                 474         Penei Sewell  
                 508            Dan Moore  
                 462       Terence Steele  
                 271        Brian O'Neill  
                 478   Christian Darrisaw  
                 61          Taylor Lewan  
                 395        Jedrick Wills  
                 378        Yosuah Nijman  
                 2       Andrew Whitworth  
                 101        Donovan Smith  
                

In [192]:
strain_data.columns.values

array(['nflId_pr', 'playId', 'frameId', 'gameId', 'assignment_dict',
       'action', 'ball_snap', 'time', 'x_pr', 'y_pr', 'officialPosition',
       'snap_time', 'nflId_qb', 'x_qb', 'y_qb', 'event', 'nflId', 'x',
       'y', 'officialPosition_pb', 'possession_id', 'dx_pr', 'dy_pr',
       'dx_qb', 'dy_qb', 'x_pr_qb', 'y_pr_qb', 'd2y_pr', 'd2x_pr',
       'pr_qb_norm', 'scalar_projection', 'd2y_pr_qb', 'd2x_pr_qb',
       'd_ij', 'strain_rate', 'strain_acceleration', 'rusher_id_model'],
      dtype=object)

In [196]:
strain_data["dy_pr_qb"] = ((
            strain_data["x_pr_qb"] * strain_data["dx_pr"]
            + strain_data["y_pr_qb"] * strain_data["dy_pr"]
        ) / strain_data["pr_qb_norm"])* strain_data["y_pr_qb"]

In [317]:
strain_data["total_strain"] = strain_data.groupby(["frameId","gameId","playId"])["strain_rate"].transform(lambda x: x.sum())

In [318]:
strain_data["num_rushers"] = strain_data.groupby(["frameId","gameId","playId"])["nflId_pr"].transform(lambda x: len(x.unique()))

In [320]:
strain_data["prop_attention"] = strain_data["assignment_dict"].apply(lambda x: sum(ast.literal_eval(x).values()))

In [321]:
strain_data["strain_created"] = (strain_data["prop_attention"]/strain_data["num_rushers"])* (strain_data["total_strain"] - strain_data["strain_rate"])

In [323]:
strain_data["total_strain_created"] = strain_data["strain_created"] + strain_data["strain_rate"]

In [337]:
strain_data.groupby(["nflId_pr"]).agg(individual_strain_created = ("strain_rate","mean"), total_strain_created = ("total_strain_created","mean")).reset_index().merge(players,left_on = "nflId_pr", right_on = "nflId").merge(snaps).groupby("officialPosition").apply(lambda x: x[(x.officialPosition=="NT") & (x.num_snaps >= 90)].sort_values("individual_strain_created",ascending=False).head(20))

nflId_pr  individual_strain_created  \
officialPosition                                            
NT               465     47917                   0.101330   
                 318     46081                   0.088268   
                 171     43316                   0.081002   
                 417     47786                   0.077648   
                 151     42559                   0.060525   
                 381     46264                   0.051976   
                 394     46487                   0.050869   
                 213     43455                   0.029106   
                 307     45226                   0.025748   
                 55      39997                   0.007289   
                 12      35485                   0.006859   
                 476     47973                  -0.011899   
                 64      40042                  -0.012600   
                 179     43332                  -0.021420   
                 290     44991                  -0.037806   

                      total_strain_created  nflId height  weight   birthDate  \
officialPosition                                                               
NT               465              0.361619  47917    6-1     312  1996-05-06   
                 318              0.244720  46081    6-4     347  1995-02-05   
                 171              0.279720  43316    6-3     314  1995-10-04   
                 417              0.252402  47786    6-3     303  1997-12-21   
                 151              0.225602  42559    6-2     305  1993-10-16   
                 381              0.242091  46264    6-4     310  1995-03-21   
                 394              0.173278  46487   5-11     310  1995-11-19   
                 213              0.187874  43455    6-3     347  1994-07-01   
                 307              0.150594  45226    6-3     345  1994-05-02   
                 55               0.228168  39997    6-3     340  1992-03-30   
                 12               0.177860  35485    6-4     329  1988-10-10   
                 476              0.091268  47973    6-5     295  1996-07-22   
                 64               0.215050  40042    6-1     336  1989-02-21   
                 179              0.086696  43332    6-4     314  1994-05-08   
                 290              0.082402  44991    6-3     311  1994-11-11   

                            collegeName officialPosition          displayName  \
officialPosition                                                                
NT               465         Washington               NT          Greg Gaines   
                 318         Washington               NT             Vita Vea   
                 171               UCLA               NT          Kenny Clark   
                 417            Alabama               NT     Quinnen Williams   
                 151               Rice               NT  Christian Covington   
                 381            Rutgers               NT     Sebastian Joseph   
                 394              Texas               NT           Poona Ford   
                 213            Clemson               NT          D.J. Reader   
                 307           Colorado               NT           Josh Tupou   
                 55          Ohio State               NT    Johnathan Hankins   
                 12       East Carolina               NT        Linval Joseph   
                 476           Arkansas               NT          Armon Watts   
                 64   Missouri Southern               NT     Brandon Williams   
                 179         Penn State               NT       Austin Johnson   
                 290    Louisiana State               NT       Davon Godchaux   

                      num_snaps  
officialPosition                 
NT               465        119  
                 318        195  
                 171        238  
                 417        152  
                 151        123  
       

In [343]:
blocker_df.merge(snaps_block).merge(players).groupby("officialPosition").apply(lambda x: np.average(x.estimate, weights=x.num_snaps))

officialPosition
C      0.316216
FB     0.424440
G      0.341781
RB     1.363381
T      0.411706
TE     1.487580
WR    11.497551
dtype: float64

In [521]:
blocker_full = blocker_coef_df.merge(snaps_block).merge(players)

In [522]:
avg_blocker_full = blocker_full.groupby("officialPosition").apply(lambda x: np.average(x.estimate, weights=x.num_snaps)).reset_index().merge(blocker_full)

In [523]:
avg_blocker_full.rename(axis = 1, mapper = {0:"pos_estimate"}, inplace = True)

In [524]:
avg_blocker_full

,officialPosition,pos_estimate,index,estimate,nflId,num_snaps,height,weight,birthDate,collegeName,displayName
0,C,0.017391,betas[7],3.793718e-04,41069,55,6-2,300,1989-06-05,Iowa,James Ferentz
1,C,0.017391,betas[33],5.560349e-04,53491,115,6-5,312,NaN,NaN,Josh Myers
2,C,0.017391,betas[34],5.645388e-05,53492,339,6-5,316,NaN,NaN,Creed Humphrey
3,C,0.017391,betas[44],7.591441e-04,37130,143,6-2,315,1989-07-12,Florida State,Rodney Hudson
4,C,0.017391,betas[45],4.262140e-04,53516,228,6-3,310,NaN,NaN,Kendrick Green
...,...,...,...,...,...,...,...,...,...,...,...
529,WR,0.000127,betas[479],3.626307e-13,44817,1,6-3,209,1995-01-11,Western Michigan,Corey Davis
530,WR,0.000127,betas[485],3.052414e-12,38696,2,6-2,198,1990-03-12,California,Marvin Jones
531,WR,0.000127,betas[500],5.289224e-13,44881,2,6-2,208,1993-06-15,Eastern Washington,Cooper Kupp
532,WR,0.000127,betas[504],5.954136e-12,44896,1,6-1,209,1996-02-27,Penn State,Chris Godwin


In [525]:
avg_blocker_full["index"] = avg_blocker_full["nflId"].apply(lambda x: blocker_encode_inverse_map[x])

In [526]:
new_blocker_coefs = avg_blocker_full.sort_values("index")["pos_estimate"].values

In [472]:
X_mat = (-.5*strain_data["strain_rate"].values[:,np.newaxis]**2)*blocker_design_matrix

In [527]:
strain_data["expected_drag_acceleration"] = X_mat.dot(new_blocker_coefs)

In [528]:
strain_data["expected_acceleration"] = strain_data["expected_drag_acceleration"] + strain_data["strain_acceleration"]

In [475]:
from scipy.integrate import cumtrapz

In [529]:
strain_data["expected_strain_rate"] = strain_data.groupby(["nflId_pr","gameId","playId"])["expected_acceleration"].transform(lambda x: cumtrapz(x,dx = .1,initial = 0))

In [530]:
strain_data["total_expected_strain"] = strain_data.groupby(["frameId","gameId","playId"])["expected_strain_rate"].transform(lambda x: x.sum())

In [531]:
strain_data["expected_strain_created"] = (strain_data["prop_attention"]/strain_data["num_rushers"])* (strain_data["total_expected_strain"] - strain_data["expected_strain_rate"])

In [532]:
strain_data["total_expected_strain_created"] = strain_data["expected_strain_created"] + strain_data["expected_strain_rate"]

In [538]:
strain_data.groupby(["nflId_pr"]).agg(individual_strain_created = ("expected_strain_rate","mean"), total_strain_created = ("total_expected_strain_created","mean")).reset_index().merge(players,left_on = "nflId_pr", right_on = "nflId").merge(snaps).groupby("officialPosition").apply(lambda x: x[(x.officialPosition=="OLB") & (x.num_snaps >= 90)].sort_values("total_strain_created",ascending=False).head(20))

nflId_pr  individual_strain_created  \
officialPosition                                            
OLB              80      41231                   0.210951   
                 255     44842                   0.223411   
                 123     42381                   0.206089   
                 165     43298                   0.212454   
                 18      37075                   0.218948   
                 551     52510                   0.213549   
                 26      37145                   0.210485   
                 13      35493                   0.173255   
                 131     42401                   0.182573   
                 423     47795                   0.215014   
                 37      38551                   0.184730   
                 626     53460                   0.192585   
                 485     48089                   0.181224   
                 545     52492                   0.171401   
                 36      38548                   0.162605   
                 621     53447                   0.169728   
                 588     52650                   0.170482   
                 20      37087                   0.167320   
                 420     47790                   0.190131   
                 467     47934                   0.164190   

                      total_strain_created  nflId height  weight   birthDate  \
officialPosition                                                               
OLB              80               0.321347  41231    6-3     269  1991-02-22   
                 255              0.316305  44842    6-4     252  1994-10-11   
                 123              0.310904  42381    6-5     265  1992-11-17   
                 165              0.309799  43298    6-5     240  1992-09-08   
                 18               0.306781  37075    6-3     250  1989-03-26   
                 551              0.306038  52510    6-4     242  1997-08-07   
                 26               0.305996  37145    6-3     270  1989-01-21   
                 13               0.293809  35493    6-6     285  1989-02-28   
                 131              0.290504  42401    6-3     260  1991-03-13   
                 423              0.286208  47795    6-5     277  1997-12-03   
                 37               0.277250  38551    6-5     265  1990-02-27   
                 626              0.273788  53460    6-5     252         NaN   
                 485              0.273736  48089    6-2     235  1996-08-05   
                 545              0.271142  52492    6-5     252  1998-08-25   
                 36               0.264269  38548    6-2     247  1989-04-26   
                 621              0.262833  53447    6-5     266         NaN   
                 588              0.255369  52650    6-4     250  1999-07-28   
                 20               0.253694  37087    6-4     257  1990-05-18   
                 420              0.249694  47790    6-5     262  1997-07-13   
                 467              0.244829  47934    6-4     242  1995-07-01   

                                   collegeName officialPosition  \
officialPosition                                                  
OLB              80                    Buffalo              OLB   
                 255                 Wisconsin              OLB   
                 123         Mississippi State              OLB   
                 165                   Georgia              OLB   
                 18              Texas A&amp;M              OLB   
                 551  North Carolina-Charlotte              OLB   
                 26                    Georgia              OLB   
                 13                    Florida              OLB   
                 131                  Missouri              OLB   
                 423                  Michigan              OLB   
                 37                   Syracuse              OLB   
                 626                       NaN 

In [498]:
coef = pd.read_csv("apm_fit.csv")

,index,estimate
0,betas[1],0.000338
1,betas[2],0.000631
2,betas[3],0.000422
3,betas[4],0.000042
4,betas[5],0.000388
...,...,...
1228,betas_rusher[695],0.000003
1229,betas_rusher[696],0.169666
1230,betas_rusher[697],0.031083
1231,betas_rusher[698],0.336528


In [500]:
coef_df = coef.iloc[0][1:].reset_index()

In [501]:
coef_df.rename(axis = 1, inplace=True, mapper = {0:"estimate"})

In [503]:
rusher_coef_df = coef_df[coef_df["index"].str.contains("rusher")]

In [504]:
rusher_coef_df["nflId"] = rusher_coef_df["index"].apply(lambda x: rusher_encoder_map[int(x.replace("[",".").replace("]","").split(".")[-1]) -1])

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_95379/754334621.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rusher_coef_df["nflId"] = rusher_coef_df["index"].apply(lambda x: rusher_encoder_map[int(x.replace("[",".").replace("]","").split(".")[-1]) -1])


In [506]:
rusher_coef_df

,index,estimate,nflId
534,betas_rusher[1],0.047591,45063
535,betas_rusher[2],0.053161,43062
536,betas_rusher[3],0.011204,53315
537,betas_rusher[4],0.056562,39008
538,betas_rusher[5],0.000696,45203
...,...,...,...
1227,betas_rusher[694],0.063842,45021
1228,betas_rusher[695],0.000003,45033
1229,betas_rusher[696],0.169666,45038
1230,betas_rusher[697],0.031083,45042


In [510]:
rusher_coef_df.merge(players).merge(snaps, left_on = ["nflId"], right_on = ["nflId_pr"]).groupby(["officialPosition"]).apply(lambda x: x[(x.num_snaps >= 90) & (x.officialPosition =="DT")].sort_values("estimate",ascending=False).head(20))

index  estimate  nflId height  weight  \
officialPosition                                                          
DT               582  betas_rusher[583]  0.103954  38542    6-4     310   
                 36    betas_rusher[37]  0.101167  37104    6-5     295   
                 216  betas_rusher[217]  0.097739  47802    6-4     300   
                 545  betas_rusher[546]  0.097617  42443    6-4     318   
                 212  betas_rusher[213]  0.095531  47796    6-4     315   
                 405  betas_rusher[406]  0.094907  52415    6-5     318   
                 163  betas_rusher[164]  0.092995  43441    6-3     310   
                 394  betas_rusher[395]  0.091312  46249    6-4     318   
                 22    betas_rusher[23]  0.090120  53467    6-5     310   
                 318  betas_rusher[319]  0.088455  39959    6-3     294   
                 136  betas_rusher[137]  0.080125  41341    6-4     322   
                 107  betas_rusher[108]  0.074727  43356    6-2     308   
                 308  betas_rusher[309]  0.074032  46082    6-3     320   
                 186  betas_rusher[187]  0.069296  43638    6-3     292   
                 344  betas_rusher[345]  0.069278  46144    6-1     310   
                 539  betas_rusher[540]  0.069071  52665    6-3     290   
                 568  betas_rusher[569]  0.058480  42560    6-2     307   
                 94    betas_rusher[95]  0.056020  43338    6-3     306   
                 499  betas_rusher[500]  0.052358  42375    6-2     320   
                 64    betas_rusher[65]  0.037301  43301    6-2     305   

                       birthDate           collegeName officialPosition  \
officialPosition                                                          
DT               582  1990-12-13     Mississippi State               DT   
                 36   1989-05-06            Ohio State               DT   
                 216  1997-07-28     Mississippi State               DT   
                 545  1992-11-14                Auburn               DT   
                 212  1995-12-20               Clemson               DT   
                 405  1998-04-15                Auburn               DT   
                 163  1994-01-11                Temple               DT   
                 394  1995-03-04           Connecticut               DT   
                 22          NaN                   NaN               DT   
                 318  1990-11-29              Missouri               DT   
                 136  1991-12-27            Penn State               DT   
                 107  1995-04-08              Nebraska               DT   
                 308  1997-05-27               Alabama               DT   
                 186  1992-10-23          Ferris State               DT   
                 344  1996-05-09         Florida State               DT   
                 539  1998-06-09                 Texas               DT   
                 568  1993-07-03  Southern Mississippi               DT   
                 94   1992-12-16               Alabama               DT   
                 499  1994-02-02                 Texas               DT   
                 64   1994-04-02            Louisville               DT   

                              displayName  nflId_pr  num_snaps  
officialPosition                                                
DT               582         Fletcher Cox     38542        175  
                 36       Cameron Heyward     37104        198  
                 216      Jeffery Simmons     47802        263  
                 545      Angelo Blackson     42443        110  
                 212    Christian Wilkins     47796        153  
                 405        Derrick Brown     52415        158  
                 163       Matt Ioannidis     43441        146  
                 394   Folorunso Fatukasi     46249        121  
                 22     Christian Barmore     53467        183  
                 

In [513]:
blocker_coef_df = coef_df[ (coef_df["index"].str.contains("betas[",regex=False))]

In [514]:
blocker_coef_df

,index,estimate
0,betas[1],3.384342e-04
1,betas[2],6.312900e-04
2,betas[3],4.220361e-04
3,betas[4],4.183526e-05
4,betas[5],3.875585e-04
...,...,...
529,betas[530],1.092695e-13
530,betas[531],2.371780e-13
531,betas[532],9.506997e-13
532,betas[533],2.749758e-13


In [460]:
blocker_coef_df = blocker_coef_df.iloc[1:]

In [429]:
blocker_design_matrix.shape

(1150270, 534)

In [430]:
blocker_design_matrix = pd.read_csv("blocker_rusher_design_matrix.csv").values

In [432]:
X_mat = (-.5*strain_data["strain_rate"].values[:,np.newaxis]**2)*blocker_design_matrix

In [434]:
len(strain_data)

1150270

In [448]:
len(blocker_encode_map)

534

In [437]:
blocker_encode_map = {index:val for index,val in enumerate(set(chain.from_iterable([list(ast.literal_eval(item).keys()) for item in strain_data["assignment_dict"]])))}

In [438]:
len(blocker_encode_map)

534

In [516]:
blocker_coef_df["nflId"] = blocker_coef_df["index"].apply(lambda x: blocker_encode_map[int(x.replace("[",".").replace("]","").split(".")[-1]) -1])

/var/folders/d4/h18vf05j4jsgmvd2j5k8nl840000gn/T/ipykernel_95379/3854537891.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blocker_coef_df["nflId"] = blocker_coef_df["index"].apply(lambda x: blocker_encode_map[int(x.replace("[",".").replace("]","").split(".")[-1]) -1])


In [520]:
blocker_coef_df.merge(players).merge(snaps_block).groupby("officialPosition").apply(lambda x: x[(x.num_snaps >=90) & (x.officialPosition == "C")].sort_values("estimate",ascending=False).head(20))

index      estimate  nflId height  weight  \
officialPosition                                                       
C                121  betas[122]  5.097312e-01  43433    6-4     306   
                 116  betas[117]  7.940853e-04  37266    6-3     295   
                 43    betas[44]  7.591441e-04  37130    6-2     315   
                 85    betas[86]  7.067987e-04  41293    6-6     315   
                 32    betas[33]  5.560349e-04  53491    6-5     312   
                 44    betas[45]  4.262140e-04  53516    6-3     310   
                 163  betas[164]  3.376786e-04  47801    6-3     305   
                 137  betas[138]  2.908244e-04  43510    6-4     305   
                 123  betas[124]  2.619818e-04  41390    6-3     301   
                 63    betas[64]  2.399415e-04  43307    6-4     309   
                 160  betas[161]  2.115808e-04  43695    6-3     313   
                 131  betas[132]  1.996612e-04  41436    6-3     300   
                 158  betas[159]  1.056196e-04  41630    6-3     317   
                 155  betas[156]  8.898505e-05  41619    6-3     316   
                 33    betas[34]  5.645388e-05  53492    6-5     316   
                 242  betas[243]  9.126539e-11  46090    6-4     312   
                 401  betas[402]  9.016133e-11  42392    6-6     305   
                 193  betas[194]  5.667306e-11  47861    6-6     310   
                 372  betas[373]  4.514284e-11  52554    6-3     320   
                 343  betas[344]  1.093199e-11  52486    6-4     290   

                       birthDate           collegeName officialPosition  \
officialPosition                                                          
C                121  1993-04-27              Missouri                C   
                 116  1987-11-05            Cincinnati                C   
                 43   1989-07-12         Florida State                C   
                 85   1991-05-29              Missouri                C   
                 32          NaN                   NaN                C   
                 44          NaN                   NaN                C   
                 163  1995-06-20  North Carolina State                C   
                 137  1993-03-15              Illinois                C   
                 123  1991-07-27            Ohio State                C   
                 63   1993-05-30               Alabama                C   
                 160  1993-02-17                  Duke                C   
                 131  1989-10-12           Boise State                C   
                 158  1992-05-31       Central Florida                C   
                 155  1992-07-06                 Texas                C   
                 33          NaN                   NaN                C   
                 242  1994-10-11            Ohio State                C   
                 401  1992-04-21              Missouri                C   
                 193  1996-09-03             Wisconsin                C   
                 372  1997-11-20             Wisconsin                C   
                 343  1997-11-17                Temple                C   

                           displayName  num_snaps  
officialPosition                                   
C                121   Connor McGovern        271  
                 116       Jason Kelce        255  
                 43      Rodney Hudson        143  
                 85       Justin Britt        175  
                 32         Josh Myers        115  
                 44     Kendrick Green        228  
                 163  Garrett Bradbury        254  
                 137        Ted Karras        143  
                 123     Corey Linsley        274  
                 63         Ryan Kelly        258  
                 160        Matt Skura        142  
                 131      Matt Paradis        294  
                 158     Justin McCray         97  
           

{0: 45056,
 1: 45062,
 2: 45069,
 3: 43045,
 4: 45094,
 5: 45142,
 6: 41069,
 7: 30842,
 8: 30869,
 9: 45217,
 10: 53433,
 11: 53436,
 12: 53442,
 13: 53443,
 14: 39109,
 15: 53446,
 16: 53452,
 17: 53453,
 18: 45267,
 19: 45268,
 20: 53464,
 21: 37082,
 22: 53466,
 23: 53471,
 24: 45281,
 25: 37090,
 26: 53475,
 27: 53480,
 28: 53482,
 29: 39146,
 30: 53484,
 31: 37101,
 32: 53491,
 33: 53492,
 34: 53497,
 35: 53499,
 36: 37118,
 37: 53506,
 38: 41222,
 39: 53510,
 40: 53512,
 41: 45321,
 42: 53514,
 43: 37130,
 44: 53516,
 45: 53517,
 46: 41232,
 47: 53522,
 48: 53523,
 49: 41236,
 50: 41237,
 51: 53524,
 52: 53527,
 53: 41242,
 54: 45339,
 55: 43293,
 56: 43295,
 57: 53536,
 58: 43297,
 59: 45346,
 60: 53539,
 61: 43302,
 62: 53543,
 63: 43307,
 64: 39212,
 65: 53549,
 66: 41262,
 67: 45355,
 68: 41264,
 69: 53553,
 70: 53555,
 71: 53556,
 72: 53557,
 73: 43320,
 74: 37179,
 75: 43324,
 76: 43329,
 77: 53571,
 78: 43334,
 79: 41286,
 80: 53574,
 81: 53576,
 82: 43337,
 83: 53579,
 8